## LSH

In [ ]:
# =================== BOOTSTRAP CELL ===================
# Standard setup for all notebooks
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]  # assumes notebooks are in a subfolder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ========================================================
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# from src.config.variables import COVARIATES

# ========================================================
# Optional for warnings and nicer plots
import warnings
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

import sys
from pathlib import Path

# ========================================================
# Ensuring project root is in the Python path
# Adjusting this if your notebooks are nested deeper
PROJECT_ROOT = Path.cwd().parents[0]  # assumes notebooks are in a subfolder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ========================================================
# Importing helper to load paths
from src.utils.helpers import load_paths

# ========================================================
# Loading paths from config.yaml (works regardless of notebook location)
paths = load_paths()

# ========================================================
for key, value in paths.items():
    print(f"{key}: {value}")

# ========================================================
# Now you can use these paths in your notebook:
SYNTHETIC_DATA_DIR = paths['SYNTHETIC_DATA_DIR']
PROCESSED_DATA_DIR = paths['PROCESSED_DATA_DIR']
# FIG_DIR = paths['FIG_DIR']

# ========================================================

### Load data

In [ ]:
import pandas as pd

df_A = pd.read_pickle(PROCESSED_DATA_DIR / "df_A_encoded.pkl")
df_B = pd.read_pickle(PROCESSED_DATA_DIR / "df_B_encoded.pkl")

### Bloom Filters to Arrays

In [1]:
import numpy as np

def bf_to_array(bf_string):
    return np.array([int(x) for x in bf_string])

### LSH Function

In [2]:
import hashlib

def lsh_blocking(df, bands=20, rows_per_band=5):

    blocks = {}

    for idx, row in df.iterrows():

        bf = bf_to_array(row["bloom"])

        for b in range(bands):
            start = b * rows_per_band
            end = start + rows_per_band

            band_slice = bf[start:end]

            # Convert to string
            band_str = ''.join(map(str, band_slice))

            # Hash band
            band_hash = hashlib.md5(band_str.encode()).hexdigest()

            # Create block key
            block_key = f"{b}_{band_hash}"

            if block_key not in blocks:
                blocks[block_key] = []

            blocks[block_key].append(idx)

    return blocks

### Generate candidate pairs

In [3]:
def generate_lsh_pairs(blocks_A, blocks_B):

    pairs = set()

    for key in blocks_A:
        if key in blocks_B:

            for i in blocks_A[key]:
                for j in blocks_B[key]:
                    pairs.add((i, j))

    return list(pairs)

In [ ]:
blocks_A = lsh_blocking(df_A)
blocks_B = lsh_blocking(df_B)

pairs = generate_lsh_pairs(blocks_A, blocks_B)

print("Total candidate pairs:", len(pairs))

### Compute RR - Reduction Ratio

In [ ]:
def reduction_ratio(n_pairs, n_total):
    return 1 - (n_pairs / n_total)

### Experimental Loop

In [ ]:
blocking_methods = ["rule", "lsh"]

results = []

for method in blocking_methods:

    start_time = time.time()

    if method == "rule":
        pairs = generate_rule_based_pairs(df_A, df_B)

    elif method == "lsh":
        blocks_A = lsh_blocking(df_A)
        blocks_B = lsh_blocking(df_B)
        pairs = generate_lsh_pairs(blocks_A, blocks_B)

    runtime = time.time() - start_time

    n_pairs = len(pairs)
    n_total = len(df_A) * len(df_B)

    rr = reduction_ratio(n_pairs, n_total)

    # Then run your linkage + evaluation
    matches = parallel_compare(pairs, threshold=0.85)

    # Compute precision/recall/f1 (your existing code)

    results.append({
        "method": method,
        "pairs": n_pairs,
        "reduction_ratio": rr,
        "runtime": runtime,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

### Plot

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(results)

plt.figure()
plt.scatter(df["reduction_ratio"], df["recall"])

for i, row in df.iterrows():
    plt.text(row["reduction_ratio"], row["recall"], row["method"])

plt.xlabel("Reduction Ratio")
plt.ylabel("Recall")
plt.title("Blocking Efficiency vs Accuracy")

plt.savefig("blocking_comparison.png", dpi=300)
plt.show()